# Project 01 — BROKEN notebook (debugging exercise)

This notebook contains **seeded bugs**. Your job: run it, read the diagnostics, find each bug, and fix it. The clean reference is `notebook.ipynb`; the answer key is `BROKEN_BUGS.md` (don't peek first).

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
RNG = 20240601

In [ ]:
from data.generate_data import generate
data = generate()
y = data['y']

### Model — something here over-trusts extreme rates, and the sampler is starved.

In [ ]:
# BUG 1: a 'flat' prior that is not as innocent as it looks.
# BUG 2: far too few tuning steps + 1 chain -> unreliable diagnostics.
with pm.Model() as model:
    theta = pm.Beta('theta', alpha=1.0, beta=1.0)
    pm.Bernoulli('y', p=theta, observed=y)
    idata = pm.sample(draws=1000, tune=5, chains=1, random_seed=RNG,
                      progressbar=False)

In [ ]:
print(az.summary(idata, var_names=['theta']))

### Posterior predictive — BUG 3: a wrong reduction axis makes the PPC nonsensical.

In [ ]:
with model:
    idata.extend(pm.sample_posterior_predictive(idata, random_seed=RNG,
                                                progressbar=False))
pp = idata.posterior_predictive['y']
# BUG 3: summing over the wrong dimension (chain/draw instead of observations)
pp_k = pp.sum(dim='draw').values.ravel()
print('observed k =', data['k'], 'predicted k mean =', pp_k.mean())